# smart-meter-readings: Colab runner

Edit **only the next cell** (`DRIVE_DIR`), then *Runtime → Run all*. Every run writes into `DRIVE_DIR`
(`recordings/`, `logs/`, `timing/`). Runs are resumable: after a disconnect, run all again. Complete runs are
skipped and partial runs are redone from scratch.

In [ ]:
# The ONLY cell you edit by hand: the Drive folder that receives every recording and log.
DRIVE_DIR = "/content/drive/MyDrive/smart-meter-runs"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("outputs ->", DRIVE_DIR)

In [ ]:
# Clone the repo, or pull if it is already here.
import os, subprocess
REPO_URL = "https://github.com/reemwaheed13/smart-meter-readings"
REPO_DIR = "/content/smart-meter-readings"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

In [ ]:
# Flower (with Ray) and the pinned pyyaml. Colab's own torch and numpy are kept on purpose.
%pip install -q "flwr[simulation]==1.45.0" "pyyaml==6.0.4"

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-c",
    "import sys, flwr, torch, numpy; from train.env import machine_info; "
    "print('python', sys.version.split()[0]); print('flwr', flwr.__version__); "
    "print('torch', torch.__version__); print('numpy', numpy.__version__); print(machine_info())"],
    cwd=REPO_DIR, check=True)

In [ ]:
# Tests (real Flower simulations, a few minutes).
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR)
assert r.returncode == 0, "pytest failed: fix before running experiments"

In [ ]:
# Every (level, eps, n, k) from config.yaml, into DRIVE_DIR. Resumable: rerun this cell after a disconnect.
import subprocess, sys, yaml
cfg = yaml.safe_load(open(f"{REPO_DIR}/config.yaml"))
exp, cal = cfg["experiment"], cfg["calibration"]
SMOKE = {"lr": 0.1, "T": 10, "C": 1.0, "clipping_range": 8.0}   # used only for values config.yaml leaves null

def flags(level, n):
    f = []
    if cal["lr"] is None: f += ["--lr", str(SMOKE["lr"])]
    if (cal["T"] or {}).get(n) is None: f += ["--T", str(SMOKE["T"])]
    if level in (2, 3) and cal["C"] is None: f += ["--C", str(SMOKE["C"])]
    if level == 3 and cal["clipping_range"] is None: f += ["--clipping-range", str(SMOKE["clipping_range"])]
    return f

grid = [(1, "none")] + [(2, e) for e in exp["eps"]["level2"]] + [(3, "none" if e is None else e) for e in exp["eps"]["level3"]]
failed = []
for n in exp["n"]:
    for k in exp["neighbourhoods_k"]:
        for level, eps in grid:
            extra = flags(level, n)
            if extra:
                print(f"SMOKE OVERRIDES (config.yaml not calibrated): {' '.join(extra)}")
            cmd = [sys.executable, "-m", "train.run", "--level", str(level), "--eps", str(eps),
                   "--n", str(n), "--k", str(k), "--out", DRIVE_DIR] + extra
            r = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
            lines = [l for l in r.stdout.splitlines() if l.startswith("[L")]
            print("\n".join(lines[-3:]) if lines else "")
            if r.returncode != 0:
                failed.append((level, eps, n, k))
                print(f"FAILED L{level} eps{eps} n{n} k{k}:\n" + "\n".join(r.stderr.splitlines()[-5:]))
print("failed runs:", failed or "none")

In [ ]:
# Timing test: 50 rounds of level 3, n = 20, eps = 1 -> DRIVE_DIR/timing/timing.json.
# Fake data is fine here: the cost does not depend on the readings.
import subprocess, sys, json, yaml
cfg = yaml.safe_load(open(f"{REPO_DIR}/config.yaml")); cal = cfg["calibration"]
T_proj = (cal["T"] or {}).get(20) or 10
extra = []
if cal["lr"] is None: extra += ["--lr", "0.1"]
if cal["C"] is None: extra += ["--C", "1.0"]
if cal["clipping_range"] is None: extra += ["--clipping-range", "8.0"]
r = subprocess.run([sys.executable, "-m", "train.timing_test", "--out", f"{DRIVE_DIR}/timing", "--T", str(T_proj)] + extra,
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stderr[-2000:] if r.returncode else json.dumps(json.load(open(f"{DRIVE_DIR}/timing/timing.json")), indent=2))

In [ ]:
# val_mse per round for every run in DRIVE_DIR/logs.
import glob, os, csv
import matplotlib.pyplot as plt
logs = sorted(glob.glob(f"{DRIVE_DIR}/logs/L*_nb*.csv"))
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for path in logs:
    name = os.path.basename(path)[:-4]
    level = int(name[1])
    rows = list(csv.DictReader(open(path)))
    axes[level - 1].plot([int(r["round"]) for r in rows], [float(r["val_mse"]) for r in rows], label=name, lw=1)
for i, ax in enumerate(axes):
    ax.set_title(f"level {i + 1}"); ax.set_xlabel("round"); ax.set_ylabel("val_mse"); ax.set_yscale("log")
    if ax.lines: ax.legend(fontsize=6)
plt.tight_layout(); plt.show()